![image](https://raw.githubusercontent.com/IBM/watsonx-ai-samples/master/cloud/notebooks/headers/watsonx-Prompt_Lab-Notebook.png)
# Use watsonx, and `openai/gpt-oss-120b` to analyze car rental customer satisfaction from text.

#### Disclaimers

- Use only Projects and Spaces that are available in watsonx context.


## Notebook content

This notebook contains the steps and code to demonstrate support of text sentiment analysis in watsonx. It introduces commands for data retrieval, model testing and scoring.

Some familiarity with Python is helpful. This notebook uses Python 3.12.


## Learning goal

The goal of this notebook is to demonstrate how to use `openai/gpt-oss-120b` model to analyze customer satisfaction from text.


## Contents

This notebook contains the following parts:

1. [Set up the environment](#Set-up-the-environment)
2. [Data loading](#Data-loading)
3. [Foundation Models on watsonx.ai](#Foundation-Models-on-watsonx.ai)
4. [Analyze the sentiment](#Analyze-the-sentiment)
5. [Summary and next steps](#Summary-and-next-steps)

<a id="Set-up-the-environment"></a>
## Set up the environment

Before you use the sample code in this notebook, you must perform the following setup tasks:

-  Create a <a href="https://cloud.ibm.com/catalog/services/watsonxai-runtime" target="_blank" rel="noopener no referrer">watsonx.ai Runtime Service</a> instance (a free plan is offered and information about how to create the instance can be found <a href="https://dataplatform.cloud.ibm.com/docs/content/wsj/getting-started/wml-plans.html?context=wx&audience=wdp" target="_blank" rel="noopener no referrer">here</a>).


### Install and import dependencies
**Note:** `ibm-watsonx-ai` documentation can be found <a href="https://ibm.github.io/watsonx-ai-python-sdk/index.html" target="_blank" rel="noopener no referrer">here</a>.

In [1]:
%pip install wget | tail -n 1
%pip install datasets | tail -n 1
%pip install "scikit-learn==1.6.1" | tail -n 1
%pip install -U ibm-watsonx-ai | tail -n 1

### Defining the watsonx.ai credentials
This cell defines the watsonx.ai credentials required to work with watsonx Foundation Model inferencing.

**Action:** Provide the IBM Cloud user API key. For details, see
[documentation](https://cloud.ibm.com/docs/account?topic=account-userapikey&interface=ui).

In [2]:
import getpass

from ibm_watsonx_ai import Credentials

credentials = Credentials(
    url="https://us-south.ml.cloud.ibm.com",
    api_key=getpass.getpass("Please enter your watsonx.ai api key (hit enter): "),
)

### Defining the project ID
The Foundation Model requires project ID that provides the context for the call. We will obtain the ID from the project in which this notebook runs. Otherwise, please provide the project ID.

In [3]:
import os

try:
    project_id = os.environ["PROJECT_ID"]
except KeyError:
    project_id = input("Please enter your project_id (hit enter): ")

In [4]:
from ibm_watsonx_ai import APIClient

api_client = APIClient(credentials=credentials, project_id=project_id)

<a id="Data-loading"></a>
## Data loading

Download the `car_rental_training_data` dataset. The dataset provides insight about customers opinions on car rental. It has a label that consists of values: unsatisfied, satisfied.

In [5]:
import pandas as pd
import wget

filename = "car_rental_training_data.csv"
url = "https://raw.githubusercontent.com/IBM/watsonx-ai-samples/master/cloud/data/cars-4-you/car_rental_training_data.csv"

if not os.path.isfile(filename):
    wget.download(url, out=filename)

data = pd.read_csv("car_rental_training_data.csv", sep=";")
comments = list(data.Customer_Service)
satisfaction = list(data.Satisfaction)

Examine downloaded data.

In [6]:
data.head()

,ID,Gender,Status,Children,Age,Customer_Status,Car_Owner,Customer_Service,Satisfaction,Business_Area,Action
0,83,Female,M,2,48.85,Inactive,Yes,I thought the representative handled the initi...,0,Product: Availability/Variety/Size,Free Upgrade
1,1307,Female,M,0,55.00,Inactive,No,I have had a few recent rentals that have take...,0,Product: Availability/Variety/Size,Voucher
2,1737,Male,M,0,42.35,Inactive,Yes,car cost more because I didn't pay when I rese...,0,Product: Availability/Variety/Size,Free Upgrade
3,3721,Male,M,2,61.71,Inactive,Yes,I didn't get the car I was told would be avail...,0,Product: Availability/Variety/Size,Free Upgrade
4,11,Male,S,2,56.47,Active,No,If there was not a desired vehicle available t...,1,Product: Availability/Variety/Size,NaN


Define label map.

In [7]:
label_map = {0: "unsatisfied", 1: "satisfied"}

Inspect data labels distribution. 

In [8]:
data.value_counts("Satisfaction")

Satisfaction
1    274
0    212
Name: count, dtype: int64

Prepare train and test sets.

In [9]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    data.Customer_Service,
    data.Satisfaction,
    test_size=0.3,
    random_state=33,
    stratify=data.Satisfaction,
)
data_train = pd.DataFrame(X_train)
data_test = pd.DataFrame(X_test)

data_train["satisfaction"] = pd.Series(y_train).map(label_map)
data_test["satisfaction"] = pd.Series(y_test).map(label_map)

<a id="Foundation-Models-on-watsonx.ai"></a>
## Foundation Models on watsonx.ai

#### List available models

All avaliable models are presented under TextModels class.
For more information refer to [documentation](https://ibm.github.io/watsonx-ai-python-sdk/fm_model_inference.html#TextModels).

In [10]:
api_client.foundation_models.ChatModels.show()

{'GRANITE_4_H_SMALL': 'ibm/granite-4-h-small', 'LLAMA_3_3_70B_INSTRUCT': 'meta-llama/llama-3-3-70b-instruct', 'LLAMA_4_MAVERICK_17B_128E_INSTRUCT_FP8': 'meta-llama/llama-4-maverick-17b-128e-instruct-fp8', 'MISTRAL_SMALL_3_1_24B_INSTRUCT_2503': 'mistralai/mistral-small-3-1-24b-instruct-2503', 'GPT_OSS_120B': 'openai/gpt-oss-120b'}


You need to specify `model_id` that will be used for inferencing:

In [11]:
model_id = api_client.foundation_models.ChatModels.GPT_OSS_120B

### Defining the model parameters

You might need to adjust model `parameters` for different models or tasks, to do so please refer to [documentation](https://ibm.github.io/watsonx-ai-python-sdk/fm_model.html#metanames.GenTextParamsMetaNames).

In [12]:
from ibm_watsonx_ai.metanames import GenChatParamsMetaNames as GenParams

parameters = {GenParams.TEMPERATURE: 0}

### Initialize the model
Initialize the `ModelInference` class with previous set params.

In [13]:
from ibm_watsonx_ai.foundation_models import ModelInference

model = ModelInference(
    model_id=model_id, params=parameters, credentials=credentials, project_id=project_id
)

### Model's details

In [14]:
model.get_details()

{'model_id': 'openai/gpt-oss-120b',
 'label': 'gpt-oss-120b',
 'provider': 'OpenAI',
 'source': 'Hugging Face',
 'indemnity': 'NON_IBM',
 'functions': [{'id': 'autoai_sql_rag'},
  {'id': 'text_chat'},
  {'id': 'text_generation'}],
 'short_description': 'openai/gpt-oss-120b is an OpenAI’s open-weight models designed for powerful reasoning, agentic tasks, and versatile developer use cases.',
 'long_description': 'openai/gpt-oss-120b is an OpenAI’s open-weight models designed for powerful reasoning, agentic tasks, and versatile developer use cases. It was designed for production, general purpose, high reasoning use cases with 117B parameters with 5.1B active parameters.',
 'terms_url': 'https://www.apache.org/licenses/LICENSE-2.0',
 'input_tier': 'class_8',
 'output_tier': 'class_1',
 'number_params': '120b',
 'min_shot_size': 1,
 'task_ids': ['question_answering',
  'summarization',
  'classification',
  'generation',
  'code',
  'extraction',
  'translation',
  'function_calling',
  'co

<a id="Analyze-the-sentiment"></a>
## Analyze the sentiment

Define the system prompt. 

In [15]:
system_message = {
    "role": "system",
    "content": (
        "Classify the satisfaction expressed in this sentence using one of the following options:\n"
        "- satisfied\n"
        "- unsatisfied"
    ),
}

Prepare model inputs - build zero-shot examples from the test set.

In [16]:
import json

zero_shot_inputs = [
    [system_message, {"role": "user", "content": text}]
    for text in data_test["Customer_Service"].values
]

print(json.dumps(zero_shot_inputs[:5], indent=2))

[
  [
    {
      "role": "system",
      "content": "Classify the satisfaction expressed in this sentence using one of the following options:\n- satisfied\n- unsatisfied"
    },
    {
      "role": "user",
      "content": "Provide more convenient car pickup from the airport parking."
    }
  ],
  [
    {
      "role": "system",
      "content": "Classify the satisfaction expressed in this sentence using one of the following options:\n- satisfied\n- unsatisfied"
    },
    {
      "role": "user",
      "content": "They could really try work harder."
    }
  ],
  [
    {
      "role": "system",
      "content": "Classify the satisfaction expressed in this sentence using one of the following options:\n- satisfied\n- unsatisfied"
    },
    {
      "role": "user",
      "content": "the rep was friendly but it was so loud in there that I could not hear what she was saying. I HATE having to walk across a big lot with all of my bags in search of my car which is always in the furthest corner

Prepare model inputs - build few-shot examples. To build a few-shot example few instances of training data phrases are passed together with the reference sentiment and then appended with a test data phrase. 

In this notebook, training phrases are stratified over all possible sentiments for each test case.

In [17]:
few_shot_examples = (
    data_train.groupby("satisfaction", group_keys=False)
    .apply(lambda x: x.sample(2, random_state=42))
    .reset_index(drop=True)
)

few_shot_inputs: list[list[dict]] = []

for test_phrase in data_test.Customer_Service.values:
    prompt = [system_message]

    for train_phrase, sentiment in few_shot_examples[
        ["Customer_Service", "satisfaction"]
    ].itertuples(index=False):
        prompt.append({"role": "user", "content": train_phrase})
        prompt.append({"role": "assistant", "content": sentiment})

    prompt.append({"role": "user", "content": test_phrase})
    few_shot_inputs.append(prompt)

Inspect an exemplary few-shot prompt.

In [18]:
print(json.dumps(few_shot_inputs[0], indent=2))

[
  {
    "role": "system",
    "content": "Classify the satisfaction expressed in this sentence using one of the following options:\n- satisfied\n- unsatisfied"
  },
  {
    "role": "user",
    "content": "None I  rarely rent cars"
  },
  {
    "role": "assistant",
    "content": "satisfied"
  },
  {
    "role": "user",
    "content": "they were trying to satisfy me even though my wife tells me I was being rude"
  },
  {
    "role": "assistant",
    "content": "satisfied"
  },
  {
    "role": "user",
    "content": "The counter girl seemed confused and she had lost our reservations. They directed us to one car and then came running out to tell us that was the wrong car and to come back to the counter. Getting directions took forever and they weren't accurate."
  },
  {
    "role": "assistant",
    "content": "unsatisfied"
  },
  {
    "role": "user",
    "content": "needed: shorter lines & cleaner cars, friendlier reps"
  },
  {
    "role": "assistant",
    "content": "unsatisfied"
  

### Analyze the satisfaction using chat model for zero-shot outputs

In [19]:
zero_shot_predictions = []

for prompt in zero_shot_inputs[:20]:
    zero_shot_predictions.append(model.chat(prompt)["choices"][0]["message"]["content"])

Explore model output

In [20]:
zero_shot_predictions

['unsatisfied',
 'unsatisfied',
 'unsatisfied',
 'unsatisfied',
 'satisfied',
 'satisfied',
 'unsatisfied',
 'unsatisfied',
 'unsatisfied',
 'satisfied',
 'satisfied',
 'unsatisfied',
 'unsatisfied',
 'satisfied',
 'unsatisfied',
 'satisfied',
 'satisfied',
 'unsatisfied',
 'unsatisfied',
 'unsatisfied']

Calculate accuracy

In [21]:
from sklearn.metrics import accuracy_score

accuracy_score(
    data_test["satisfaction"][: len(zero_shot_predictions)],
    zero_shot_predictions,
)

1.0

### Analyze the satisfaction using chat model for few-shot outputs

In [22]:
few_shot_predictions = []

for prompt in few_shot_inputs[:20]:
    few_shot_predictions.append(model.chat(prompt)["choices"][0]["message"]["content"])

Explore model output

In [23]:
few_shot_predictions

['unsatisfied',
 'unsatisfied',
 'unsatisfied',
 'unsatisfied',
 'satisfied',
 'satisfied',
 'unsatisfied',
 'unsatisfied',
 'unsatisfied',
 'satisfied',
 'satisfied',
 'unsatisfied',
 'unsatisfied',
 'satisfied',
 'unsatisfied',
 'satisfied',
 'satisfied',
 'unsatisfied',
 'unsatisfied',
 'unsatisfied']

Calculate accuracy

In [24]:
accuracy_score(
    data_test["satisfaction"][: len(few_shot_predictions)],
    few_shot_predictions,
)

1.0

<a id="Summary-and-next-steps"></a>
## Summary and next steps

You successfully completed this notebook!

You learned how to analyze car rental customer satisfaction with `openai/gpt-oss-120b` on watsonx. 

Check out our _[Online Documentation](https://ibm.github.io/watsonx-ai-python-sdk/samples.html)_ for more samples, tutorials, documentation, how-tos, and blog posts. 

### Authors

**Mateusz Szewczyk**, Software Engineer at watsonx.ai.

Copyright © 2023-2026 IBM. This notebook and its source code are released under the terms of the MIT License.